# 9.2 Exercise: Best Model Selection and Hyperparameter Tuning
  \
In this exercise, you will work with the Loan_Train.csv dataset which can be downloaded from this link: Loan Approval Data Set. 

Import the dataset and ensure that it loaded properly.

In [3]:
import pandas as pd

df = pd.read_csv(f"./data/loan.csv", encoding = "utf-8")
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


Drop the column “Load_ID.”

In [4]:
df = df.drop(columns = ["Loan_ID"])

Drop any rows with missing data.

In [6]:
df = df.dropna()

Convert the categorical features into dummy variables.

In [8]:
X = df.drop(columns = ["Loan_Status"])
y = df["Loan_Status"]

X = pd.get_dummies(X, drop_first = True)

y = y.map({"N": 0, "Y": 1})

print(X.shape)
print(X.head())

(480, 14)
   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   
5             5417             4196.0       267.0             360.0   

   Credit_History  Gender_Male  Married_Yes  Dependents_1  Dependents_2  \
1             1.0         True         True          True         False   
2             1.0         True         True         False         False   
3             1.0         True         True         False         False   
4             1.0         True        False         False         False   
5             1.0         True         True         False          True   

   Dependents_3+  Education_Not Graduate  Self_Employed_Yes  \
1          False                   False         

Split the data into a training and test set, where the “Loan_Status” column is the target.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (384, 14)
Test set: (96, 14)


Create a pipeline with a min-max scaler and a KNN classifier (see section 15.3 in the Machine Learning with Python Cookbook).

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

# Create pipeline
pipe = Pipeline([
    ("scaler", MinMaxScaler()),
    ("classifier", KNeighborsClassifier())
])

Fit a default KNN classifier to the data with this pipeline. Report the model accuracy on the test set. Note: Fitting a pipeline model works just like fitting a regular model.

In [17]:
knn_default = pipe.fit(X_train, y_train)
default_accuracy = knn_default.score(X_test, y_test)
print("Default KNN accuracy:", default_accuracy)

Default KNN accuracy: 0.6979166666666666


Create a search space for your KNN classifier where your “n_neighbors” parameter varies from 1 to 10. (
see section 15.3 in the Machine Learning with Python Cookbook).

In [16]:
from sklearn.model_selection import GridSearchCV

# Define search space
knn_param_grid = {
    "classifier__n_neighbors": range(1, 11)
}

# Create grid search
knn_grid = GridSearchCV(
    pipe,
    param_grid=knn_param_grid,
    cv=5,
    scoring="accuracy"
)

# Fit grid search
knn_grid.fit(X_train, y_train)
print("Best parameters:", knn_grid.best_params_)
print("Best CV accuracy:", knn_grid.best_score_)
knn_test_accuracy = knn_grid.score(X_test, y_test)

print("Best KNN test accuracy:", knn_test_accuracy)

Best parameters: {'classifier__n_neighbors': 8}
Best CV accuracy: 0.7552631578947369
Best KNN test accuracy: 0.7604166666666666


Fit a grid search with your pipeline, search space, and 5-fold cross-validation to find the best value for the “n_neighbors” parameter.

In [23]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

model_param_grid = [
    {
        "classifier": [
            LogisticRegression(
                max_iter=500,
                solver="liblinear"
            )
        ],
        "classifier__penalty": ["l1", "l2"],
        "classifier__C": np.logspace(0, 4, 10)
    },

    {
        "classifier": [
            RandomForestClassifier(random_state=0)
        ],
        "classifier__n_estimators": [10, 100, 1000],
        "classifier__max_features": [1, 2, 3]
    }
]

Find the accuracy of the grid search best model on the test set. Note: It is possible that this will not be an improvement over the default model, but likely it will be.

In [ ]:
model_grid = GridSearchCV(
    pipe,
    param_grid=model_param_grid,
    cv=5,
    scoring="accuracy"
)

model_grid.fit(X_train, y_train)

Now, repeat steps 6 and 7 with the same pipeline, but expand your search space to include logistic regression and random forest models with the hyperparameter values in section 12.3 of the Machine Learning with Python Cookbook.

In [26]:
print("Best model and parameters:")
print(model_grid.best_params_)

print("\nBest cross-validation accuracy:")
print(model_grid.best_score_)

Best model and parameters:
{'classifier': RandomForestClassifier(random_state=0), 'classifier__max_features': 3, 'classifier__n_estimators': 100}

Best cross-validation accuracy:
0.8203691045796309


What are the best model and hyperparameters found in the grid search? Find the accuracy of this model on the test set.

In [27]:
best_model_accuracy = model_grid.score(X_test, y_test)

print("Best model test accuracy:", best_model_accuracy)

Best model test accuracy: 0.75


**Summarize your results.**

The default KNN classifier achieved a test accuracy of 69.79%. After searching over values of n_neighbors from 1 through 10, the optimal value was 8, which increased the test accuracy to 76.04%.

The second grid search compared logistic regression and random forest models using the hyperparameter ranges from Recipe 12.3 of Machine Learning with Python Cookbook. The best model according to the 5-fold cross-validation score was a random forest with 100 trees and max_features=3. It achieved a cross-validation accuracy of approximately 82.04%.

Interestingly, its test accuracy was 75.00%, which is slightly lower than the tuned KNN's 76.04%. This is not contradictory: the model-selection criterion was the 5-fold cross-validation accuracy on the training data, whereas the final accuracy is measured on the held-out test set. The assignment explicitly notes that the grid-search model does not necessarily have to outperform the other model on the test set.

One important point to mention in your write-up is that the random forest was selected as the best model by cross-validation, even though tuned KNN happened to perform slightly better on this particular test split. That is the correct interpretation of the exercise.

The Cookbook's structure confirms that Recipe 12.3 is specifically about selecting the best model from multiple learning algorithms, while Recipe 15.3 addresses selecting the optimal KNN neighborhood size.```

What is the difference between a parameter and a hyperparameter of a model?

What Python libraries are useful for hyperparameter tuning?

What is cross-validation? Why is this useful?

Is a validation set the same as a test set?

What are some key hyperparameters in various machine learning models?